In [1]:
# File Settings
INPUT_FILE = "/content/drive/MyDrive/input/input.mp4"

# this is the output file that will me exported to a "RVE" folder under the "My Drive" folder in Google Drive. WILL OVERWRITE ANY EXISTING FILES
OUTPUT_FILE = "/content/output.mp4" # mkv is recommended, as it supports more codecs {mkv, mp4, avi, mov}

KILL_RUNTIME_AFTER_RENDER = True # Terminates the runtime to save on your colab compute. If copying to drive fails, manual intervention is required to download files.
# Interpolate Settings

# https://github.com/TNTwise/real-video-enhancer-models/releases/tag/models/ is where all model files for RVE are stored
# you can download them from there, or try custom models from https://openmodeldb.info/
INTERPOLATE = False # {True, False}
INTERPOLATE_FACTOR = 1
SCENE_DETECT_METHOD = "none" # {pyscenedetect, none}
SCENE_DETECT_SENSATIVITY = "3.5" # {0 - 9.9} lower is more sensative

"""
Internal Interpolate Models:
rife4.6.pkl - Speed
rife4.7.pkl - Smoothness
rife4.18.pkl - IRL
rife4.22.pkl - Animation
rife4.25.pkl - General
GMFSS.pkl - Animation
GMFSS_PRO.pkl - Animation
GIMMVFI_RAFT.pth - IRL
"""
INTERPOLATE_MODEL = "none" # {Internal model, (link to model path)}


# Upscale Settings
UPSCALE = True # {True, False}
"""
4xNomos8k_span_otf_weak.pth - Realistic 4x High Quality Input
4xNomos8k_span_otf_medium.pth - Realistic 4x Medium Quality Input
4xNomos8k_span_otf_strong.pth - Realistic 4x Low Quality Input
2x_BHI_SpanPlusDynamic_Light.pth - Realistic 2x High Quality Input (pytorch only, experimental)

2x_ModernSpanimationV2.pth - Animation 2x
2x_ModernSpanimationV3.pth - Animation 2x (pytorch only, experimental)
2x_AnimeJaNai_V2_Compact_36k.pth - Animation 2x
2x_AnimeJaNai_HD_V3_Sharp1_Compact_430k.pth - Animation 2x
up2x-latest-conservative.pth - Animation 2x (pytorch only, slow)
up3x-latest-conservative.pth - Animation 3x (pytorch only, slow)
up4x-latest-conservative.pth - Animation 4x (pytorch only, slow)
"""
UPSCALE_MODEL = "2x_AnimeJaNai_HD_V3_Sharp1_Compact_430k.pth"
OVERRIDE_UPSCALE_SCALE = 1

# FFMpeg Settings
VIDEO_ENCODER = "libx264"
AUDIO_ENCODER = "copy_audio"
SUBTITLE_ENCODER = "copy_subtitle"

CUSTOM_ENCODER = "-c:v libx264 -profile:v high -pix_fmt yuv420p -preset veryslow -crf 18 -tune animation -x264-params bframes=16:merange=32:aq-mode=3:psy-rd=0.55,0.05 -c:a copy -c:s copy"

# Backend Settings
BACKEND = "pytorch" # {pytorch, tensorrt} TensorRT is fastest, but takes a long time to install deps.


In [2]:
# will move output file to drive when done for permident storage
import os
import subprocess
import shutil
from google.colab import drive, output

# mount drive and move output there for permadent storage

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!rm -rf real-video-enhancer/
!git clone https://github.com/tntwise/real-video-enhancer

Cloning into 'real-video-enhancer'...
remote: Enumerating objects: 29185, done.
remote: Counting objects: 100% (268/268), done.
remote: Compressing objects: 100% (126/126), done.
remote: Total 29185 (delta 156), reused 188 (delta 130), pack-reused 28917 (from 3)
Receiving objects: 100% (29185/29185), 23.17 MiB | 32.87 MiB/s, done.
Resolving deltas: 100% (21438/21438), done.


In [4]:
import os
!mkdir real-video-enhancer/models/
origdir = os.getcwd()
os.chdir("real-video-enhancer/models/")

def has_model(model: str):
    return model and model.lower() != "none"

def download_model(model: str):
    os.system(f"wget https://github.com/TNTwise/real-video-enhancer-models/releases/download/models/{model}")



# get interpolate model by extracting the end of the link
def download_external_model(model_link : str):
        print("Downloading Model")
        os.system(f"wget {model_link}")
        model = model_link.split(r'/')[-1]
        return model


if INTERPOLATE:
    if "http" in INTERPOLATE_MODEL:
        UPSCALE_MODEL = download_external_model(INTERPOLATE_MODEL)
    else:
        download_model(INTERPOLATE_MODEL)
if UPSCALE:

    if "http" in UPSCALE_MODEL:
        print("up")
        UPSCALE_MODEL = download_external_model(UPSCALE_MODEL)
    else:
        download_model(UPSCALE_MODEL)


os.chdir(origdir)



In [5]:
!mkdir backend # create backend dir so rve doesnt crash. This is to prevent rve from not detecting a "backend" folder, and being unable to write to a non existant log file location.

In [6]:
if BACKEND.lower() == 'tensorrt':
    !wget https://github.com/astral-sh/python-build-standalone/releases/download/20250317/cpython-3.12.9+20250317-x86_64-unknown-linux-gnu-install_only.tar.gz
    !tar xvf cpython-3.12.9+20250317-x86_64-unknown-linux-gnu-install_only.tar.gz
    !./python/bin/python3 -m pip install  --extra-index-url https://download.pytorch.org/whl/cu126 -r real-video-enhancer/backend/requirements.txt # only install if using tensorrt

!mkdir bin/ && cd bin && wget https://github.com/TNTwise/real-video-enhancer-models/releases/download/models/ffmpeg && chmod +x ffmpeg

--2026-05-19 02:50:15--  https://github.com/TNTwise/real-video-enhancer-models/releases/download/models/ffmpeg
Resolving github.com (github.com)... 140.82.116.4
Connecting to github.com (github.com)|140.82.116.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/837362765/87cad8f2-ed9d-4334-a81c-096918b3aac8?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-05-19T03%3A23%3A30Z&rscd=attachment%3B+filename%3Dffmpeg&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-05-19T02%3A22%3A48Z&ske=2026-05-19T03%3A23%3A30Z&sks=b&skv=2018-11-09&sig=K496TxvmznC%2FbrAL5GBIpecg9TS9ieHm44maIY9SF8A%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3OTE2MjYxNSwibmJmIjoxNzc5MTU5MDE1LCJwYXRoIjoicmVsZWFzZWFzc2V0cHJvZHVjdGlvbi5ibG9

In [7]:
#example upscale/denoise command
# It is recommended to use  --video_encoder_preset x264_nvenc
def build_backend_command() -> str:
    command = (('python3 real-video-enhancer/backend/rve-backend.py' if BACKEND == "pytorch" else './python/bin/python3 real-video-enhancer/backend/rve-backend.py')
               + f' -i "{INPUT_FILE}"'
               + f' -o "{OUTPUT_FILE}"'
               + f' -b {BACKEND} '
               + f' --video_encoder_preset {VIDEO_ENCODER}'
               + f' --audio_encoder_preset {AUDIO_ENCODER}'
               + f' --subtitle_encoder_preset {SUBTITLE_ENCODER}')
    if UPSCALE_MODEL and UPSCALE:
        command += f" --upscale_model real-video-enhancer/models/{UPSCALE_MODEL} "
    if INTERPOLATE_MODEL and INTERPOLATE:
        command += f" --interpolate_model real-video-enhancer/models/{INTERPOLATE_MODEL} "
        command += f" --interpolate_factor {INTERPOLATE_FACTOR} "
        command += f" --scene_detect_method {SCENE_DETECT_METHOD} "
        command += f" --scene_detect_threshold {SCENE_DETECT_SENSATIVITY} "

    return command

with open("command.sh", "w") as f:
  f.write(build_backend_command())

!bash command.sh

BACKEND: Getting Input Video Properties
Duration: 196.58333333333334 seconds
Total Frames: 9436
Resolution: 1920x1080
FPS: 48.0
Color Space: None
Color Transfer: None
Color Primaries: None
Pixel Format: yuv420p
Video Codec: h264
Video Bitrate: 4531 kbps
Is HDR: False
Bit Depth: 8
RVE Backend Version: 2.4.1-dev16
CLI Arguments: 
['real-video-enhancer/backend/rve-backend.py', '-i', '/content/drive/MyDrive/input/input.mp4', '-o', '/content/output.mp4', '-b', 'pytorch', '--video_encoder_preset', 'libx264', '--audio_encoder_preset', 'copy_audio', '--subtitle_encoder_preset', 'copy_subtitle', '--upscale_model', 'real-video-enhancer/models/2x_AnimeJaNai_HD_V3_Sharp1_Compact_430k.pth']
BACKEND: No Working Directory specified, using current directory: /content
BACKEND: Getting Input Video Properties
BACKEND: Using backend: pytorch
BACKEND: Setting up Upscale
BACKEND: Handling device: auto, GPU ID: 0
Using Device: Tesla T4
BACKEND: Handling precision: auto
BACKEND: Initializing stream for device

In [ ]:
# Source file path in Colab
source_path = OUTPUT_FILE

# Destination path on Google Drive
destination_path = f'/content/drive/MyDrive/RVE/{os.path.basename(OUTPUT_FILE)}'

if not os.path.exists(os.path.dirname(destination_path)):
  os.makedirs(os.path.dirname(destination_path), exist_ok=True)

# Move the file
try:
  shutil.copy(source_path, destination_path)
  print(f"Copied to drive: {destination_path}")

except Exception as e:
  print(f"Copying to drive failed! {e}\nPlease download any files that have correctly rendered before terminating the runtime!.")
  input("Press enter to terminate the runtime.")

  output.eval_js('google.colab.kernel.disconnect()')
  os.kill(os.getpid(), 9)

if KILL_RUNTIME_AFTER_RENDER:

  # kill the runtime
  output.eval_js('google.colab.kernel.disconnect()')
  os.kill(os.getpid(), 9)

Copied to drive: /content/drive/MyDrive/RVE/output.mp4
